# Técnicas de Prompting — Cómo hablarle bien a un LLM

> **Objetivo:** Dominar las principales técnicas de prompt engineering para obtener respuestas más precisas, consistentes y útiles de los LLMs.

## ¿Por qué importa cómo escribes el prompt?

El mismo modelo con prompts diferentes puede darte:
- Una respuesta inútil y vaga
- Una respuesta precisa y accionable

El **prompt engineering** es la habilidad de comunicarte bien con el modelo. No es magia — son técnicas concretas y aprendibles.

## Técnicas que veremos:
| Técnica | Descripción corta | Cuándo usarla |
|---------|-------------------|---------------|
| **Zero-Shot** | Sin ejemplos | Tareas simples y claras |
| **Few-Shot** | Con 2-5 ejemplos | Necesitas un formato específico |
| **Chain-of-Thought (CoT)** | Pide razonamiento paso a paso | Problemas de lógica o matemáticas |
| **Few-Shot CoT** | Ejemplos + razonamiento | Problemas complejos con formato |
| **Self-Consistency** | Pregunta N veces y vota | Decisiones críticas |
| **Role Prompting** | El modelo adopta un rol | Perspectiva específica |
| **Negative Prompting** | Di qué NO hacer | Evitar comportamientos no deseados |

---

## 0. Preparación

In [ ]:
!pip install langchain langchain-google-genai langchain-core -q

In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="C:/Users/Oscar/OneDrive - FM4/Escritorio/EVOLVE/Data Science/EVOLVE/Victor_Prior_IA_Generativa/Proyecto_Modulo_IAGen/.env")
API_KEY = os.getenv("Gemini_API_Key_001")

MODEL = "gemini-2.5-flash-lite"
llm = ChatGoogleGenerativeAI(model=MODEL, temperature=0.5, google_api_key=API_KEY)

print("✅ Entorno listo")

c:\Users\Oscar\OneDrive - FM4\Escritorio\EVOLVE\Data Science\.venv_tf\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Entorno listo


## 1. Zero-Shot Prompting

**Zero-shot** = cero ejemplos. Solo le das la instrucción.

El modelo usa su conocimiento de entrenamiento para inferir qué quieres.

### Cuándo funciona bien:
- Tareas que el modelo ya conoce bien (clasificación, traducción, resumen)
- Cuando no tienes ejemplos disponibles
- Prototipado rápido

### Versión básica vs. detallada:
La versión **detallada** añade criterios y restricciones de formato. Siempre es mejor.

In [3]:
TAREA = "Clasifica la siguiente reseña como POSITIVA, NEGATIVA o MIXTA."
RESENA = "La cámara del móvil es increíble, pero la batería dura muy poco y el precio es abusivo."

# Zero-shot básico: instrucción mínima → respuesta impredecible en formato
zero_shot_basico = ChatPromptTemplate.from_messages([
    ("system", TAREA),
    ("human", "{resena}")
])

# Zero-shot detallado: instrucción + criterios + formato esperado → mucho más consistente
zero_shot_detallado = ChatPromptTemplate.from_messages([
    ("system", f"""{TAREA}

Criterios:
- POSITIVA: el cliente está satisfecho en general
- NEGATIVA: el cliente está insatisfecho en general
- MIXTA: hay tanto aspectos positivos como negativos

Responde SOLO con una palabra: POSITIVA, NEGATIVA o MIXTA."""),  # ← Restricción de formato
    ("human", "{resena}")
])

print("=== ZERO-SHOT DETALLADO ===")
resp1 = (zero_shot_basico | llm | StrOutputParser()).invoke({"resena": RESENA})
print(resp1)  # Puede responder con texto explicativo además de la clasificación

print("\n=== ZERO-SHOT BÁSICO ===")
resp2 = (zero_shot_detallado | llm | StrOutputParser()).invoke({"resena": RESENA})
print(resp2)  # Debería responder solo "MIXTA"

=== ZERO-SHOT DETALLADO ===
La reseña es **MIXTA**.

**Justificación:**

*   **Positiva:** Menciona que la cámara del móvil es "increíble".
*   **Negativa:** Señala que la batería "dura muy poco" y que el precio es "abusivo".

Al haber puntos positivos y negativos claros, la clasificación es mixta.

=== ZERO-SHOT BÁSICO ===
MIXTA


## 2. Few-Shot Prompting

**Few-shot** = pocos ejemplos (normalmente 2-5). Le muestras al modelo pares de `(entrada → salida)` para que aprenda el patrón.

### Por qué funciona:
El modelo infiere el patrón de los ejemplos y lo aplica a la nueva entrada.

### Ventajas sobre zero-shot:
- Más consistente en el formato de salida
- Mejor en casos ambiguos (el modelo ve cómo resolviste casos similares)
- No necesitas describir el criterio con palabras — los ejemplos lo muestran

### Estructura en LangChain:
```
System prompt
   ↓
Human: Ejemplo 1  →  AI: Respuesta 1
Human: Ejemplo 2  →  AI: Respuesta 2
...n ejemplos...
Human: NUEVA PREGUNTA  →  (el modelo completa aquí)
```

In [4]:
# Los ejemplos son pares entrada→salida que el modelo usará como referencia
ejemplos_clasificacion = [
    {
        "resena": "Producto excelente, superó mis expectativas. Envío rapidísimo.",
        "clasificacion": "POSITIVA"
    },
    {
        "resena": "Una porquería. Se rompió al día siguiente. Servicio de atención al cliente nefasto.",
        "clasificacion": "NEGATIVA"
    },
    {
        "resena": "El diseño es bonito y el precio razonable, pero la calidad del material es mejorable.",
        "clasificacion": "MIXTA"
    },
    {
        "resena": "No es lo que esperaba. Las fotos engañan mucho.",
        "clasificacion": "NEGATIVA"
    },
]

# ejemplo_prompt define cómo se formatea CADA ejemplo individual
# Cada ejemplo se convierte en: HumanMessage(reseña) → AIMessage(clasificación)
ejemplo_prompt = ChatPromptTemplate.from_messages([
    ("human", "{resena}"),
    ("ai", "{clasificacion}")
])

# FewShotChatMessagePromptTemplate ensambla todos los ejemplos
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=ejemplo_prompt,
    examples=ejemplos_clasificacion
)

# Template final: instrucción + ejemplos + nueva pregunta
template_few_shot = ChatPromptTemplate.from_messages([
    ("system", "Eres un clasificador de sentimientos de reseñas. Responde SOLO con POSITIVA, NEGATIVA o MIXTA."),
    few_shot_prompt,     # Aquí se insertan los 4 ejemplos como mensajes
    ("human", "{resena}")  # La nueva reseña a clasificar
])

# Muestra cómo queda el prompt completo con todos los mensajes
print("Estructura del prompt con ejemplos:")
messages = template_few_shot.format_messages(resena="NUEVA RESEÑA")
for i, msg in enumerate(messages):
    print(f"  [{i}] {msg.__class__.__name__}: {msg.content[:60]}...")

Estructura del prompt con ejemplos:
  [0] SystemMessage: Eres un clasificador de sentimientos de reseñas. Responde SO...
  [1] HumanMessage: Producto excelente, superó mis expectativas. Envío rapidísim...
  [2] AIMessage: POSITIVA...
  [3] HumanMessage: Una porquería. Se rompió al día siguiente. Servicio de atenc...
  [4] AIMessage: NEGATIVA...
  [5] HumanMessage: El diseño es bonito y el precio razonable, pero la calidad d...
  [6] AIMessage: MIXTA...
  [7] HumanMessage: No es lo que esperaba. Las fotos engañan mucho....
  [8] AIMessage: NEGATIVA...
  [9] HumanMessage: NUEVA RESEÑA...


In [5]:
# Comparativa en casos difíciles: reseñas ambiguas donde zero-shot puede vacilar
resenas_dificiles = [
    "No está mal para el precio que tiene.",           # Ambigua → ¿MIXTA o POSITIVA?
    "Exactamente lo que esperaba, ni más ni menos.",   # Ambigua → ¿POSITIVA o NEUTRA?
    "El producto OK pero el vendedor es un desastre.", # Mixta clara
]

cadena_zero = zero_shot_detallado | llm | StrOutputParser()
cadena_few  = template_few_shot  | llm | StrOutputParser()

print("Comparativa Zero-Shot vs Few-Shot en casos difíciles:\n")
print(f"{'Reseña':<50} {'Zero-Shot':<12} {'Few-Shot':<12}")
print("-" * 74)

for resena in resenas_dificiles:
    zs = cadena_zero.invoke({"resena": resena}).strip()
    fs = cadena_few.invoke({"resena": resena}).strip()
    print(f"'{resena[:47]}'  {zs:<12} {fs:<12}")

Comparativa Zero-Shot vs Few-Shot en casos difíciles:

Reseña                                             Zero-Shot    Few-Shot    
--------------------------------------------------------------------------
'No está mal para el precio que tiene.'  MIXTA        MIXTA       
'Exactamente lo que esperaba, ni más ni menos.'  POSITIVA     POSITIVA    
'El producto OK pero el vendedor es un desastre.'  MIXTA        MIXTA       


## 3. Chain-of-Thought (CoT) Prompting

**CoT** = pides al modelo que **muestre su razonamiento** antes de dar la respuesta final.

### ¿Por qué mejora la precisión?
El modelo no genera la respuesta de golpe — la "construye" paso a paso. Cada paso usa el anterior como contexto, lo que reduce errores de cálculo y saltos lógicos.

### Activar CoT:
- **Zero-Shot CoT** (más simple): añadir `"Piensa paso a paso."` al prompt
- **CoT con formato**: especificar el formato exacto de los pasos
- **Few-Shot CoT**: dar ejemplos de razonamiento completo

### Cuándo no usarlo:
- Tareas simples (añade tokens innecesarios = mayor coste)
- Clasificación directa donde el razonamiento no ayuda

In [6]:
PROBLEMA = """
En una empresa hay 3 departamentos: Ventas (15 personas), Marketing (8 personas) y TI (12 personas).
El 40% de Ventas trabaja de forma remota.
El 75% de Marketing trabaja de forma remota.
El 50% de TI trabaja de forma remota.
¿Cuántas personas en total trabajan de forma presencial?
"""

# Sin CoT: el modelo intenta responder directamente → más propenso a errores
sin_cot = ChatPromptTemplate.from_messages([
    ("system", "Responde la pregunta directamente con el número."),
    ("human", "{problema}")
])

# Con CoT: el modelo sigue un formato de pasos → menos errores
con_cot = ChatPromptTemplate.from_messages([
    ("system", """Resuelve el problema paso a paso.

Usa este formato:
PASO 1: [primer cálculo]
PASO 2: [segundo cálculo]
...
RESPUESTA FINAL: [resultado]"""),
    ("human", "{problema}")
])

print("=== SIN CHAIN-OF-THOUGHT ===")
resp_sin = (sin_cot | llm | StrOutputParser()).invoke({"problema": PROBLEMA})
print(resp_sin)

print("\n\n=== CON CHAIN-OF-THOUGHT ===")
resp_con = (con_cot | llm | StrOutputParser()).invoke({"problema": PROBLEMA})
print(resp_con)

# Nota: la respuesta correcta es:
# Ventas presencial: 15 * 0.60 = 9
# Marketing presencial: 8 * 0.25 = 2
# TI presencial: 12 * 0.50 = 6
# Total: 9 + 2 + 6 = 17

=== SIN CHAIN-OF-THOUGHT ===
27


=== CON CHAIN-OF-THOUGHT ===
PASO 1: Calcular el número de personas que trabajan de forma remota en Ventas.
Número de personas en Ventas = 15
Porcentaje de personas que trabajan de forma remota en Ventas = 40%
Personas que trabajan de forma remota en Ventas = 15 * 0.40 = 6

PASO 2: Calcular el número de personas que trabajan de forma presencial en Ventas.
Personas que trabajan de forma presencial en Ventas = Total de personas en Ventas - Personas que trabajan de forma remota en Ventas
Personas que trabajan de forma presencial en Ventas = 15 - 6 = 9

PASO 3: Calcular el número de personas que trabajan de forma remota en Marketing.
Número de personas en Marketing = 8
Porcentaje de personas que trabajan de forma remota en Marketing = 75%
Personas que trabajan de forma remota en Marketing = 8 * 0.75 = 6

PASO 4: Calcular el número de personas que trabajan de forma presencial en Marketing.
Personas que trabajan de forma presencial en Marketing = Total de pe

In [7]:
# Zero-Shot CoT: la forma más simple de activar CoT
# Solo añades "Piensa paso a paso" al prompt — sin cambiar la estructura
zero_shot_cot = ChatPromptTemplate.from_messages([
    ("human", "{problema}\n\nPiensa paso a paso antes de dar la respuesta.")
    #                      ↑ Esta frase sola activa el comportamiento CoT en la mayoría de modelos
])

print("=== ZERO-SHOT CoT (solo 'piensa paso a paso') ===")
resp = (zero_shot_cot | llm | StrOutputParser()).invoke({"problema": PROBLEMA})
print(resp)

=== ZERO-SHOT CoT (solo 'piensa paso a paso') ===
Aquí tienes el razonamiento paso a paso para resolver el problema:

1.  **Entender el objetivo:** Queremos saber cuántas personas trabajan de forma *presencial* en total en la empresa.

2.  **Identificar la información clave:**
    *   Total de personas en Ventas: 15
    *   Total de personas en Marketing: 8
    *   Total de personas en TI: 12
    *   Porcentaje de Ventas que trabaja remoto: 40%
    *   Porcentaje de Marketing que trabaja remoto: 75%
    *   Porcentaje de TI que trabaja remoto: 50%

3.  **Determinar la información necesaria para el objetivo:** Para saber cuántos trabajan presencial, necesitamos saber cuántos *no* trabajan remoto. Si conocemos el porcentaje que trabaja remoto, podemos calcular el porcentaje que trabaja presencial.

4.  **Calcular el porcentaje de personas que trabajan presencial en cada departamento:**
    *   **Ventas:** Si el 40% trabaja remoto, entonces el 100% - 40% = 60% trabaja presencial.
    *   

## 4. Few-Shot CoT — Ejemplos + Razonamiento

Combina lo mejor de few-shot (ejemplos) y CoT (razonamiento). Los ejemplos incluyen el razonamiento completo, no solo la respuesta final.

### Cuándo usar Few-Shot CoT:
- Problemas complejos donde el razonamiento requiere un formato específico
- Cuando Zero-Shot CoT no es suficientemente consistente

In [8]:
# Los ejemplos muestran el razonamiento completo, no solo la respuesta
ejemplos_cot = [
    {
        "problema": "Una tienda tiene 50 camisetas. Vende el 30% el lunes y el 20% de las restantes el martes. ¿Cuántas le quedan?",
        "solucion": """RAZONAMIENTO:
- Lunes: 50 × 0.30 = 15 vendidas → quedan 50 - 15 = 35
- Martes: 35 × 0.20 = 7 vendidas → quedan 35 - 7 = 28
RESPUESTA: 28 camisetas"""
    },
    {
        "problema": "Si 5 trabajadores construyen una pared en 8 días, ¿cuánto tardarán 10 trabajadores?",
        "solucion": """RAZONAMIENTO:
- Trabajo total = 5 × 8 = 40 días-trabajador
- Con 10 trabajadores: 40 / 10 = 4 días
RESPUESTA: 4 días"""
    }
]

# Los ejemplos muestran al modelo exactamente cómo debe razonar Y presentar la respuesta
ejemplo_prompt_cot = ChatPromptTemplate.from_messages([
    ("human", "{problema}"),
    ("ai", "{solucion}")
])

few_shot_cot = FewShotChatMessagePromptTemplate(
    example_prompt=ejemplo_prompt_cot,
    examples=ejemplos_cot
)

template_final_cot = ChatPromptTemplate.from_messages([
    ("system", "Eres un matemático experto. Siempre muestras tu razonamiento antes de dar la respuesta."),
    few_shot_cot,
    ("human", "{problema}")
])

resp = (template_final_cot | llm | StrOutputParser()).invoke({"problema": PROBLEMA})
print("=== FEW-SHOT CoT ===")
print(resp)

=== FEW-SHOT CoT ===
RAZONAMIENTO:
1.  **Calcular el número de personas que trabajan de forma remota en cada departamento:**
    *   Ventas: 15 personas * 40% = 15 * 0.40 = 6 personas trabajan de forma remota.
    *   Marketing: 8 personas * 75% = 8 * 0.75 = 6 personas trabajan de forma remota.
    *   TI: 12 personas * 50% = 12 * 0.50 = 6 personas trabajan de forma remota.

2.  **Calcular el número total de personas que trabajan de forma remota:**
    *   Total remoto = 6 (Ventas) + 6 (Marketing) + 6 (TI) = 18 personas.

3.  **Calcular el número total de personas en la empresa:**
    *   Total de empleados = 15 (Ventas) + 8 (Marketing) + 12 (TI) = 35 personas.

4.  **Calcular el número de personas que trabajan de forma presencial:**
    *   Personas presenciales = Total de empleados - Total remoto
    *   Personas presenciales = 35 - 18 = 17 personas.

RESPUESTA:
17 personas trabajan de forma presencial.


## 5. Self-Consistency — Fiabilidad por Votación

**Self-consistency** = preguntas N veces (con temperatura alta para variabilidad) y te quedas con la **respuesta más frecuente** (mayoría de votos).

### ¿Por qué funciona?
Si el modelo llega al mismo resultado por diferentes caminos, esa respuesta es más probable que sea correcta.

### Cuándo usar:
- Decisiones críticas donde el coste del error es alto
- Preguntas ambiguas donde el modelo vacila
- Cuando necesitas una métrica de confianza

### Cuándo NO usar:
- Cuando el coste de API importa (haces N llamadas en lugar de 1)
- Tareas donde la variabilidad es deseada (escritura creativa)

In [ ]:
from collections import Counter

# Temperatura alta para que cada llamada produzca respuestas variadas
# Si temperatura=0, todas las respuestas serían iguales → sin valor en self-consistency
llm_variado = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0.8,
    google_api_key=API_KEY
)

def self_consistency(template, inputs: dict, n_muestras: int = 5, extractor=None):
    """
    Ejecuta la cadena n_muestras veces y devuelve la respuesta por mayoría.
    
    Args:
        template: el ChatPromptTemplate a usar
        inputs: variables del template
        n_muestras: cuántas veces preguntar al modelo
        extractor: función opcional para extraer la respuesta final de cada output
    
    Returns:
        dict con respuesta_consenso, votos, confianza y todas las respuestas
    """
    cadena = template | llm_variado | StrOutputParser()
    
    # Genera n_muestras respuestas independientes
    respuestas = [cadena.invoke(inputs) for _ in range(n_muestras)]

    # Extrae la respuesta final de cada output si hay un extractor
    respuestas_finales = [extractor(r) for r in respuestas] if extractor else [r.strip() for r in respuestas]

    # Counter cuenta cuántas veces aparece cada respuesta
    conteo = Counter(respuestas_finales)
    ganadora = conteo.most_common(1)[0][0]  # La más frecuente

    return {
        "respuesta_consenso": ganadora,
        "votos": dict(conteo),
        "confianza": conteo[ganadora] / n_muestras,  # 1.0 = unanimidad
        "todas_respuestas": respuestas
    }

template_clasificacion_simple = ChatPromptTemplate.from_messages([
    ("system", "Clasifica el sentimiento. Responde ÚNICAMENTE con: POSITIVA, NEGATIVA o MIXTA."),
    ("human", "{resena}")
])

# Reseña ambigua — el modelo puede vacilar entre respuestas
resena_ambigua = "No es el mejor producto que he comprado, pero tampoco el peor."

print(f"Reseña: '{resena_ambigua}'\n")
print(f"Generando {5} respuestas independientes...\n")

resultado = self_consistency(
    template_clasificacion_simple,
    {"resena": resena_ambigua},
    n_muestras=5
)

print(f"Votos: {resultado['votos']}")
print(f"Respuesta por consenso: {resultado['respuesta_consenso']}")
print(f"Confianza: {resultado['confianza']*100:.0f}%")

## 6. Role Prompting — Adoptar un Rol

Le dices al modelo que actúe como un personaje o experto concreto. Esto cambia:
- El vocabulario usado
- La perspectiva desde la que responde
- El nivel de detalle técnico
- Las conclusiones y prioridades

### Cuándo usarlo:
- Quieres una perspectiva específica (técnica, estratégica, crítica)
- Generar contenido para una audiencia específica
- Obtener diferentes ángulos de un mismo tema

In [9]:
# La misma pregunta con tres roles diferentes → tres respuestas radicalmente distintas
roles = [
    ("CEO",       "Eres el CEO de una empresa de IA. Responde desde una perspectiva estratégica y de negocio."),
    ("Ingeniero", "Eres un ingeniero de software senior. Responde con detalle técnico e implementación práctica."),
    ("Crítico",   "Eres un periodista crítico especializado en tecnología. Señala siempre los riesgos y limitaciones."),
]

PREGUNTA_ROL = "¿Cuál es el impacto de los LLMs en el mercado laboral?"

for rol, system in roles:
    template_rol = ChatPromptTemplate.from_messages([
        ("system", system),
        ("human", "{pregunta}")
    ])
    resp = (template_rol | llm | StrOutputParser()).invoke({"pregunta": PREGUNTA_ROL})
    print(f"\n🎭 [{rol}]")
    print("-" * 40)
    print(resp[:400])  # Solo mostramos los primeros 400 caracteres para comparar


🎭 [CEO]
----------------------------------------
Como CEO de una empresa de IA, veo el impacto de los Modelos de Lenguaje Grandes (LLMs) en el mercado laboral no como una amenaza singular, sino como una **transformación multifacética y profunda** que requiere una estrategia proactiva y adaptativa. Mi enfoque se centra en cómo podemos, como industria y como sociedad, navegar esta transición para maximizar los beneficios y mitigar los riesgos.

Aq

🎭 [Ingeniero]
----------------------------------------
Como ingeniero de software senior, el impacto de los Modelos de Lenguaje Grandes (LLMs) en el mercado laboral es un tema que me apasiona y que observo con gran interés. No se trata de una simple tendencia tecnológica, sino de una **transformación fundamental** que está redefiniendo roles, creando nuevas oportunidades y, sí, también generando desafíos significativos.

Aquí te presento un análisis d

🎭 [Crítico]
----------------------------------------
## El Impacto de los LLMs en el Mercad

## 7. Negative Prompting — Decir qué NO Hacer

En lugar de (o además de) decir qué quieres, especificas qué quieres **evitar**.

### Por qué funciona:
Los modelos a veces tienen comportamientos por defecto (usar superlativos, jerga corporativa, frases hechas) que son difíciles de eliminar solo con instrucciones positivas. Las restricciones negativas son más directas.

### Patrón:
```
NO hagas:
- [comportamiento 1 a evitar]
- [comportamiento 2 a evitar]

SÍ haz:
- [comportamiento 1 deseado]
- [comportamiento 2 deseado]
```

In [10]:
template_sin_negativos = ChatPromptTemplate.from_messages([
    ("system", """Eres un redactor de contenido de marketing.

NO hagas lo siguiente:
- No uses superlativos vacíos como 'el mejor', 'increíble', 'revolucionario'
- No uses jerga de startup como 'disruptivo', 'game-changer', 'paradigma'
- No incluyas frases hechas como 'en el mundo actual' o 'en el panorama actual'
- No uses más de 100 palabras

SÍ haz lo siguiente:
- Usa datos concretos cuando sea posible
- Habla de beneficios específicos para el usuario
- Usa un tono directo y honesto"""),
    ("human", "Escribe un párrafo de presentación para {producto}")
])

resp = (template_sin_negativos | llm | StrOutputParser()).invoke({
    "producto": "una plataforma de análisis de sentimientos con IA"
})
print("=== NEGATIVE PROMPTING ===")
print(resp)

=== NEGATIVE PROMPTING ===
Nuestra plataforma de análisis de sentimientos con IA te ayuda a comprender lo que tus clientes piensan realmente. Analiza miles de reseñas, comentarios y menciones en redes sociales para identificar tendencias y puntos débiles. Obtén información procesable para mejorar tus productos y servicios, aumentando la satisfacción del cliente y tu reputación.


## Resumen de Técnicas

| Técnica | Cuándo usarla | Coste extra |
|---------|---------------|-------------|
| **Zero-Shot** | Tareas claras y simples | Sin coste extra |
| **Few-Shot** | Necesitas formato específico | Tokens de los ejemplos |
| **CoT** | Problemas matemáticos o de lógica | Tokens de razonamiento |
| **Few-Shot CoT** | Problemas complejos con formato | Tokens de ejemplos + razonamiento |
| **Self-Consistency** | Decisiones críticas | N × coste de una llamada |
| **Role Prompting** | Perspectiva específica | Tokens del rol (mínimos) |
| **Negative Prompting** | Evitar comportamientos | Tokens de restricciones (mínimos) |

### Orden de prueba recomendado:
```
Zero-Shot → Zero-Shot detallado → Few-Shot → Few-Shot CoT → Self-Consistency
```
Empieza simple y añade complejidad solo si los resultados no son suficientemente buenos.